# OpenAI GPT on Amazon Bedrock — Responses API core

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

OpenAI's models on the `bedrock-mantle` endpoint. This family is the only one
with **Web Search**, **server-side tools**, and **explicit prompt-cache
breakpoints**, so it needs more coverage than the others — five notebooks
rather than one. This one covers the core Responses API; the siblings cover
the rest.

| Notebook | Covers |
|---|---|
| **01 (this one)** | Responses API core, reasoning, streaming, state, async |
| `02-web-search-and-grounding.ipynb` | Web Search, citations, IAM governance |
| `03-tools-and-structured-output.ipynb` | Client-side tools, strict JSON |
| `04-prompt-caching-and-cost.ipynb` | Explicit cache breakpoints, economics |
| `05-server-side-tools-and-fine-tuning.ipynb` | Lambda MCP (Model Context Protocol), notes/tasks, RFT (reinforcement fine-tuning), batch |

**Models covered**

| Model ID | Notes |
|---|---|
| `openai.gpt-5.6-sol` | Frontier; explicit prompt caching |
| `openai.gpt-5.6-terra` | Frontier sibling |
| `openai.gpt-5.6-luna` | Frontier sibling |
| `openai.gpt-5.5` | Previous generation; automatic caching |
| `openai.gpt-5.4` | Earlier generation |
| `openai.gpt-oss-120b` / `-20b` | Open-weight; **bare `/v1` path**, server-side tools |
| `openai.gpt-oss-safeguard-120b` / `-20b` | Safety-classification variants |

## Two path families inside one provider
This is the subtlety unique to OpenAI on mantle:

- `openai.gpt-5.*` → **`/openai/v1`**
- `openai.gpt-oss*` → **bare `/v1`**

Get it wrong and you get a 400 saying the model doesn't exist. We prove it in §2.

## Self-contained, but see also
- **Auth, the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention),
  CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retries, service tiers, TTFT (time-to-first-token)** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `redact_ids` | shortens opaque service IDs (`resp_…`, `proj_…`) before they reach committed output |
| `response_text` | assistant text from a Responses API payload |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, parse_json_lenient, post, redact_ids, response_text, safe_print

# gpt-5.5 and gpt-5.6-sol are NOT in us-west-2 for every account — us-east-1 is
# the safest choice for this family. See section 10.
REGION = "us-east-1"

SOL = "openai.gpt-5.6-sol"
TERRA = "openai.gpt-5.6-terra"
LUNA = "openai.gpt-5.6-luna"
GPT55 = "openai.gpt-5.5"
GPT54 = "openai.gpt-5.4"
OSS120 = "openai.gpt-oss-120b"
OSS20 = "openai.gpt-oss-20b"

GPT5_PREFIX = "/openai/v1"  # gpt-5.x
OSS_PREFIX = "/v1"  # gpt-oss


def prefix_for(model_id: str) -> str:
    """gpt-5.x lives under /openai/v1; gpt-oss under bare /v1."""
    return GPT5_PREFIX if model_id.startswith("openai.gpt-5.") else OSS_PREFIX


print("gpt-5.6-sol  ->", prefix_for(SOL))
print("gpt-oss-120b ->", prefix_for(OSS120))

gpt-5.6-sol  -> /openai/v1
gpt-oss-120b -> /v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "openai.gpt-5.4",
    "openai.gpt-5.5",
    "openai.gpt-5.6-luna",
    "openai.gpt-5.6-sol",
    "openai.gpt-5.6-terra",
    "openai.gpt-oss-120b",
    "openai.gpt-oss-20b",
    "openai.gpt-oss-safeguard-120b",
    "openai.gpt-oss-safeguard-20b",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
print()
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


openai.gpt-5.4                         -- not on runtime --                     mantle


openai.gpt-5.5                         -- not on runtime --                     mantle


openai.gpt-5.6-luna                    us.openai.gpt-5.6-luna                   mantle, runtime


openai.gpt-5.6-sol                     us.openai.gpt-5.6-sol                    mantle, runtime


openai.gpt-5.6-terra                   us.openai.gpt-5.6-terra                  mantle, runtime


openai.gpt-oss-120b                    openai.gpt-oss-120b-1:0                  mantle


openai.gpt-oss-20b                     openai.gpt-oss-20b-1:0                   mantle


openai.gpt-oss-safeguard-120b          openai.gpt-oss-safeguard-120b            mantle, runtime


openai.gpt-oss-safeguard-20b           openai.gpt-oss-safeguard-20b             mantle, runtime

=> 7/9 of these are on bedrock-runtime; 5 under a different id.
   bedrock-mantle only: ['openai.gpt-5.4', 'openai.gpt-5.5']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. First call

Auth is a short-term Bedrock API key minted from ambient IAM credentials: valid
≤12 h, not refreshable, Region-pinned. (`../00-foundations/01` covers the
self-refreshing provider and the SigV4 (AWS Signature Version 4) alternative that
needs no key.)

In [3]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# One client per path family, because the base URL differs.
gpt5 = OpenAI(
    api_key=provide_token(region=REGION),
    base_url=f"https://bedrock-mantle.{REGION}.api.aws{GPT5_PREFIX}",
)
oss = OpenAI(
    api_key=provide_token(region=REGION),
    base_url=f"https://bedrock-mantle.{REGION}.api.aws{OSS_PREFIX}",
)

resp = gpt5.responses.create(
    model=SOL,
    input="Explain what an inference engine does, in two sentences.",
    max_output_tokens=200,
)
print(resp.output_text)
print("\nusage:", resp.usage.model_dump_json())

An inference engine applies logical rules to known facts to derive new conclusions or make decisions. It is a core component of expert systems, rule-based AI, and some knowledge-based applications.

usage: {"input_tokens":17,"input_tokens_details":{"cache_write_tokens":0,"cached_tokens":0},"output_tokens":40,"output_tokens_details":{"reasoning_tokens":0},"total_tokens":57}


## 2. The path split, demonstrated

Send each model to both prefixes and watch which combinations work.

In [4]:
print(f"{'model':24} {'/openai/v1':>12} {'/v1':>8}")
print("-" * 46)
for model in (SOL, GPT55, GPT54, OSS120, OSS20):
    row = {}
    for label, prefix in (("/openai/v1", GPT5_PREFIX), ("/v1", OSS_PREFIX)):
        code, _ = post(
            f"{prefix}/responses",
            {"model": model, "input": "Reply OK", "max_output_tokens": 16},
            region=REGION,
        )
        row[label] = code
    print(f"{model:24} {row['/openai/v1']:>12} {row['/v1']:>8}")

model                      /openai/v1      /v1
----------------------------------------------


openai.gpt-5.6-sol                200      400


openai.gpt-5.5                    200      400


openai.gpt-5.4                    200      400


openai.gpt-oss-120b               400      200


openai.gpt-oss-20b                400      200


Two clean groups: `gpt-5.*` is served only under `/openai/v1`, and `gpt-oss*` only
under bare `/v1`. Send a model to the other prefix and it 400s.

That is a question about the **path**. Underneath it sits a different question — which
**API**, and which **parameter** — and a bare 400 does not tell the two apart.


## 2b. Which API, and which output-cap parameter

Two questions that look like one. A 400 from Chat Completions can mean *this model
does not serve Chat Completions* or *this model does not accept the parameter you
sent* — and the fix is completely different.

Every OpenAI model here serves **both** Responses and Chat Completions. What differs
is the name of the output cap:

| | Responses | Chat Completions |
|---|---|---|
| gpt-5.6 (`sol`, `terra`, `luna`) | `max_output_tokens` only | `max_completion_tokens` only |
| gpt-5.5, gpt-5.4, gpt-oss | either | either |

The GPT-5.6 family rejects `max_tokens` with *"Unsupported parameter: 'max_tokens' is
not supported with this model"*. Send the wrong one and you get a 400 that looks like
the API is missing.

This is the same shape of trap as the 12 August 2026 Gemma 4 change, where
`max_tokens` stopped being accepted on Chat Completions. When a request that worked
yesterday starts returning 400 `unsupported_parameter`, suspect the parameter before
you suspect the endpoint.


In [5]:
# Vary the API *and* the parameter name, so a 400 tells you which one is at fault.
CAPS = {
    "/responses": ("max_output_tokens", "max_tokens"),
    "/chat/completions": ("max_completion_tokens", "max_tokens"),
}

print(f"{'model':24} {'API':18} {'documented':>11} {'max_tokens':>11}")
print("-" * 68)
for model in (SOL, TERRA, LUNA, GPT55, GPT54, OSS120):
    prefix = prefix_for(model)
    for path, (documented, legacy) in CAPS.items():
        codes = []
        for field in (documented, legacy):
            body = {"model": model, field: 16}
            if path == "/responses":
                body["input"] = "Reply OK"
            else:
                body["messages"] = [{"role": "user", "content": "Reply OK"}]
            code, _ = post(f"{prefix}{path}", body, region=REGION)
            codes.append(code)
        print(f"{model:24} {path:18} {codes[0]:>11} {codes[1]:>11}")

print("\n200 in the 'documented' column means the API is available.")
print("400 in the 'max_tokens' column is a parameter problem, not a missing API.")


model                    API                 documented  max_tokens
--------------------------------------------------------------------


openai.gpt-5.6-sol       /responses                 200         400


openai.gpt-5.6-sol       /chat/completions          200         400


openai.gpt-5.6-terra     /responses                 200         400


openai.gpt-5.6-terra     /chat/completions          200         400


...

openai.gpt-5.5           /chat/completions          200         200


openai.gpt-5.4           /responses                 200         200


openai.gpt-5.4           /chat/completions          200         200


openai.gpt-oss-120b      /responses                 200         200


openai.gpt-oss-120b      /chat/completions          200         200

200 in the 'documented' column means the API is available.
400 in the 'max_tokens' column is a parameter problem, not a missing API.


## 3. Sampling parameters

Which sampling parameters a model accepts is per-model, and it changes — Gemma 4's
surface tightened overnight in August 2026 (`../03-google-gemma/01` §3). So the
cell below probes rather than asserts.

The transferable lesson is the one the output makes obvious: **do not share a single
sampling config across models.** Two models from the same provider can disagree, and
`../03-google-gemma/` shows the split running the other way.

In [6]:
for model in (SOL, OSS120):
    prefix = prefix_for(model)
    print(f"\n{model}")
    for label, extra in [
        ("temperature=1.0", {"temperature": 1.0}),
        ("top_p=0.95", {"top_p": 0.95}),
    ]:
        code, data = post(
            f"{prefix}/responses",
            {"model": model, "input": "Reply OK", "max_output_tokens": 16, **extra},
            region=REGION,
        )
        print(f"   {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")


openai.gpt-5.6-sol


   temperature=1.0    -> HTTP 200 


   top_p=0.95         -> HTTP 400 Unsupported parameter: 'top_p' is not supported with this mo

openai.gpt-oss-120b


   temperature=1.0    -> HTTP 200 


   top_p=0.95         -> HTTP 200 


In [7]:
# max_output_tokens has a MINIMUM of 16 on the Responses API.
for n in (8, 16):
    code, data = post(
        f"{GPT5_PREFIX}/responses",
        {"model": SOL, "input": "Hi", "max_output_tokens": n},
        region=REGION,
    )
    detail = "" if code == 200 else err(data)[:70]
    print(f"max_output_tokens={n:3} -> HTTP {code} {detail}")

max_output_tokens=  8 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


max_output_tokens= 16 -> HTTP 200 


## 4. Reasoning

Effort ladder is `none` / `low` / `medium` / `high` — `minimal` is rejected.
The reasoning trace is returned as separate `reasoning` items in `output`, which
is a Responses-API-only capability.

In [8]:
for effort in ("none", "minimal", "low", "medium", "high"):
    code, data = post(
        f"{GPT5_PREFIX}/responses",
        {
            "model": SOL,
            "input": "2+2?",
            "max_output_tokens": 32,
            "reasoning": {"effort": effort},
        },
        region=REGION,
    )
    print(f"  effort={effort:8} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  effort=none     -> HTTP 200 


  effort=minimal  -> HTTP 400 Unsupported value: 'minimal' is not supported with the 'open


  effort=low      -> HTTP 200 


  effort=medium   -> HTTP 200 


  effort=high     -> HTTP 200 


In [9]:
reasoned = gpt5.responses.create(
    model=SOL,
    input=(
        "Three switches outside a room control one bulb inside. You may flip "
        "switches freely but enter the room only once. How do you identify the "
        "correct switch?"
    ),
    reasoning={"effort": "high"},
    max_output_tokens=1500,
)
print("output item types:", [i.type for i in reasoned.output])
print("reasoning tokens :", reasoned.usage.output_tokens_details.reasoning_tokens)
print("\n=== ANSWER ===")
print(reasoned.output_text[:400])

output item types: ['reasoning', 'message']
reasoning tokens : 54

=== ANSWER ===
1. Turn on switch 1 for several minutes, then turn it off.  
2. Turn on switch 2 and enter the room.  
3. Check the bulb:
   - **Lit:** switch 2 controls it.
   - **Off but warm:** switch 1 controls it.
   - **Off and cold:** switch 3 controls it.


### Effort changes spend — but only when the task needs thinking

The obvious experiment gives a misleading answer, so run both halves. On an easy
question this model spends **zero** reasoning tokens at every effort level: effort
is a ceiling, not a floor, and the model decides. Only on a task that actually
needs working-out does the setting show up in the bill.

In [10]:
EASY = "What is 17 * 23? Answer with the number only."
HARD = (
    "Five houses in a row, each a different colour, each owner a different "
    "nationality and pet. The Brit lives in the red house. The Swede keeps dogs. "
    "The Dane drinks tea. The green house is immediately left of the white house. "
    "The green owner drinks coffee. Who owns the fish? Nationality only."
)

print(f"{'task':6} {'effort':8} {'reasoning tok':>14} {'output tok':>11} {'latency':>9}")
print("-" * 54)
spend = {}
for label, prompt in (("easy", EASY), ("hard", HARD)):
    for effort in ("none", "low", "medium", "high"):
        started = time.perf_counter()
        r = gpt5.responses.create(
            model=SOL,
            input=prompt,
            reasoning={"effort": effort},
            # Generous: a tight cap on a reasoning model returns status
            # "incomplete" and an empty answer, which would confound this table.
            max_output_tokens=3000,
        )
        elapsed = time.perf_counter() - started
        tokens = r.usage.output_tokens_details.reasoning_tokens
        spend[(label, effort)] = tokens
        print(
            f"{label:6} {effort:8} {tokens:>14} "
            f"{r.usage.output_tokens:>11} {elapsed:>8.2f}s"
        )

# Derived, because the single claim this cell used to make ("effort changes spend")
# was false on the easy prompt and non-monotonic on the hard one.
easy = [spend[("easy", e)] for e in ("none", "low", "medium", "high")]
hard = [spend[("hard", e)] for e in ("none", "low", "medium", "high")]
print()
if not any(easy):
    print("easy task: 0 reasoning tokens at every effort — the model judged it did")
    print("           not need to think. Raising effort cost nothing.")
if hard[0] == 0 and any(hard[1:]):
    print('hard task: effort="none" spent 0, the rest spent reasoning tokens, so the')
    print("           knob does work — but read the numbers, not the ordering:")
    print(f"           {dict(zip(('none', 'low', 'medium', 'high'), hard))}")
    if hard[1:] != sorted(hard[1:]):
        print("           low/medium/high are NOT in order from one sample each.")
        print("           Take medians before costing a change.")

task   effort    reasoning tok  output tok   latency
------------------------------------------------------


easy   none                  0           5     0.69s


easy   low                   0           5     0.75s


easy   medium                0           5     0.70s


easy   high                  0           5     0.70s


hard   none                  0          25     0.85s


hard   low                  85         115     2.31s


hard   medium              248         282     4.49s


hard   high                230         244     4.21s

easy task: 0 reasoning tokens at every effort — the model judged it did
           not need to think. Raising effort cost nothing.
hard task: effort="none" spent 0, the rest spent reasoning tokens, so the
           knob does work — but read the numbers, not the ordering:
           {'none': 0, 'low': 85, 'medium': 248, 'high': 230}
           low/medium/high are NOT in order from one sample each.
           Take medians before costing a change.


## 5. Verbosity and other gpt-5.6 controls

The frontier models accept several extra Responses parameters. Probe them so you
know what is safe to send.

In [11]:
extras = [
    ("text.verbosity=low", {"text": {"verbosity": "low"}}),
    ("truncation=auto", {"truncation": "auto"}),
    ("metadata", {"metadata": {"team": "samples"}}),
    ("max_tool_calls=2", {"max_tool_calls": 2}),
    ("parallel_tool_calls", {"parallel_tool_calls": True}),
    (
        "include reasoning.encrypted_content",
        {"include": ["reasoning.encrypted_content"]},
    ),
]
for label, extra in extras:
    code, data = post(
        f"{GPT5_PREFIX}/responses",
        {"model": SOL, "input": "Reply OK", "max_output_tokens": 16, **extra},
        region=REGION,
    )
    print(f"  {label:38} -> HTTP {code} {'' if code == 200 else err(data)[:50]}")

  text.verbosity=low                     -> HTTP 200 


  truncation=auto                        -> HTTP 200 


  metadata                               -> HTTP 200 


  max_tool_calls=2                       -> HTTP 200 


  parallel_tool_calls                    -> HTTP 200 


  include reasoning.encrypted_content    -> HTTP 200 


In [12]:
# Verbosity has a visible effect on answer length.
for verbosity in ("low", "high"):
    r = gpt5.responses.create(
        model=SOL,
        input="What is a load balancer?",
        text={"verbosity": verbosity},
        max_output_tokens=500,
    )
    print(
        f"verbosity={verbosity:5} -> {len(r.output_text):4} chars | "
        f"{r.output_text[:80]!r}"
    )

verbosity=low   ->  500 chars | 'A **load balancer** distributes incoming network traffic across multiple servers'


verbosity=high  -> 2169 chars | 'A **load balancer** is a system that distributes incoming network or application'


## 6. Streaming

Reasoning and answer text arrive on distinct event types, so you can render a
"thinking" panel separately from the answer.

In [13]:
stream = gpt5.responses.create(
    model=SOL,
    input="List three trade-offs of microservice architectures.",
    reasoning={"effort": "low"},
    max_output_tokens=500,
    stream=True,
)
counts = {}
print("--- live ---")
try:
    for event in stream:
        counts[event.type] = counts.get(event.type, 0) + 1
        if event.type == "response.reasoning_text.delta":
            print("\033[2m" + event.delta + "\033[0m", end="", flush=True)
        elif event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after the deltas above: {type(exc).__name__}]")
print("\n\n--- event types ---")
for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")

--- live ---


1

.

 **

Independent

...

 developer

-training

 costs

.



--- event types ---
   108  response.output_text.delta
     1  response.created
     1  response.in_progress
     1  response.output_item.added
     1  response.content_part.added
     1  response.output_text.done
     1  response.content_part.done
     1  response.output_item.done
     1  response.completed


## 7. Multi-turn: history array vs server-side state

In [14]:
conversation = [
    {"role": "system", "content": "You are terse."},
    {"role": "user", "content": "What is a circuit breaker in distributed systems?"},
]
first = gpt5.responses.create(model=SOL, input=conversation, max_output_tokens=200)
print("assistant:", first.output_text[:160])

conversation += [
    {"role": "assistant", "content": first.output_text},
    {"role": "user", "content": "What is the usual half-open state for?"},
]
second = gpt5.responses.create(model=SOL, input=conversation, max_output_tokens=200)
print("\nassistant:", second.output_text[:160])

assistant: A **circuit breaker** is a resilience pattern that prevents repeated calls to a failing or slow remote service.

It typically has three states:

- **Closed:** R



assistant: The **half-open state** tests whether the failed service has recovered.

After the open-state cooldown, the circuit breaker allows a limited number of trial req


**Do not replay reasoning items** into the next turn — send only final answers.
Keep reasoning in your own logs.

In [15]:
# Server-side state: cheaper on input tokens, but requires store=True which
# retains input and output for 30 days in-Region (see ../00-foundations/02).
turn1 = gpt5.responses.create(
    model=SOL,
    input="My deploy tool is CodeDeploy. Reply: noted.",
    max_output_tokens=32,
    store=True,
)
turn2 = gpt5.responses.create(
    model=SOL,
    input="Which deploy tool did I mention?",
    previous_response_id=turn1.id,
    max_output_tokens=48,
)
print("chained recall:", turn2.output_text[:120])

code, data = post(
    f"{GPT5_PREFIX}/responses",
    {
        "model": SOL,
        "input": "And again?",
        "max_output_tokens": 32,
        "previous_response_id": gpt5.responses.create(
            model=SOL, input="secret. reply ok", max_output_tokens=16, store=False
        ).id,
    },
    region=REGION,
)
print(f"chaining from store=False -> HTTP {code}: {err(data)[:80]}")

chained recall: CodeDeploy.


chaining from store=False -> HTTP 404: Response not found.


## 8. Async / background jobs

Long jobs can run detached and be polled — useful for deep-reasoning tasks that
would otherwise hold an HTTP connection open.

In [16]:
bg = gpt5.responses.create(
    model=SOL,
    input="Write three paragraphs on the CAP theorem and its practical limits.",
    max_output_tokens=800,
    background=True,
    store=True,
)
safe_print("job:", bg.id, "status:", bg.status)

for _ in range(40):
    time.sleep(3)
    code, polled = post(
        f"{GPT5_PREFIX}/responses/{bg.id}", None, region=REGION, method="GET"
    )
    if polled.get("status") in ("completed", "failed", "cancelled"):
        break
print("final status:", polled.get("status"))
print("text:", response_text(polled)[:220])

job: resp_mjdnp2wq... status: in_progress


final status: completed
text: The CAP theorem states that a distributed data system cannot simultaneously guarantee **consistency**, **availability**, and **partition tolerance** when a network partition occurs. Consistency means every read returns t


In [17]:
for rid in (turn1.id, bg.id):
    code, _ = post(
        f"{GPT5_PREFIX}/responses/{rid}", None, region=REGION, method="DELETE"
    )
    print(f"DELETE {redact_ids(rid)} -> {code}")

DELETE resp_xk7f772e... -> 200


DELETE resp_mjdnp2wq... -> 200


## 9. gpt-oss and the safeguard variants

The open-weight models sit at the bare `/v1` path and support both Responses and
Chat Completions. The `safeguard` variants are tuned for safety classification.

In [18]:
r = oss.responses.create(
    model=OSS120,
    input="Explain in one sentence what an open-weight model is.",
    max_output_tokens=150,
)
print("gpt-oss-120b:", r.output_text[:180])

gpt-oss-120b: An open‑weight model is a machine‑learning model whose trained parameters (weights) are publicly released, allowing anyone to download, inspect, modify, and run the model without r


In [19]:
# The safeguard models classify content against a policy you supply.
code, data = post(
    f"{OSS_PREFIX}/chat/completions",
    {
        "model": "openai.gpt-oss-safeguard-20b",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Policy: flag any request seeking personal medical advice. "
                    "Answer with exactly ALLOW or FLAG."
                ),
            },
            {
                "role": "user",
                "content": "What dosage of ibuprofen should I take for my back?",
            },
        ],
        "max_tokens": 300,
    },
    region=REGION,
)
verdict = (data.get("choices") or [{}])[0].get("message", {}).get("content", "")
print("safeguard-20b verdict:", repr((verdict or "").strip()[:120]))

safeguard-20b verdict: 'FLAG'


## 10. Regional footprint — check before you deploy

This family is *not* uniformly available. `gpt-5.5` and `gpt-5.6-sol` were absent
from `us-west-2` at the time of writing, while `gpt-oss` reaches every Region.

In [20]:
watch = [SOL, TERRA, LUNA, GPT55, GPT54, OSS120, OSS20]
regions = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
inventory = {}
for reg in regions:
    try:
        inventory[reg] = set(list_models(reg))
    except (RuntimeError, OSError) as exc:
        inventory[reg] = set()
        print(f"{reg}: {type(exc).__name__}")

print(f"{'model':32} " + "  ".join(f"{r:>13}" for r in regions))
print("-" * 92)
for model in watch:
    cells = "  ".join(
        f"{('yes' if model in inventory[r] else '-'):>13}" for r in regions
    )
    print(f"{model:32} {cells}")

model                                us-east-1      us-east-2      us-west-2   eu-central-1
--------------------------------------------------------------------------------------------
openai.gpt-5.6-sol                         yes            yes              -              -
openai.gpt-5.6-terra                       yes            yes            yes              -
openai.gpt-5.6-luna                        yes            yes            yes              -
openai.gpt-5.5                             yes            yes              -              -
openai.gpt-5.4                             yes            yes            yes              -
openai.gpt-oss-120b                        yes            yes            yes            yes
openai.gpt-oss-20b                         yes            yes            yes            yes


## 11. Compare the generations

In [21]:
task = "In one sentence, why does batching improve GPU inference throughput?"
print(f"{'model':26} {'latency':>9} {'reason tok':>11} {'out tok':>8}  answer")
print("-" * 108)
for model in (OSS20, OSS120, GPT54, GPT55, SOL):
    prefix = prefix_for(model)
    started = time.perf_counter()
    code, data = post(
        f"{prefix}/responses",
        {
            "model": model,
            "input": task,
            "max_output_tokens": 200,
            "reasoning": {"effort": "low"},
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:26} {'-':>9} {'-':>11} {'-':>8}  HTTP {code}: {err(data)[:34]}")
        continue
    usage = data.get("usage", {})
    details = usage.get("output_tokens_details", {})
    text = " ".join(response_text(data).split())
    print(
        f"{model:26} {elapsed:>8.2f}s {details.get('reasoning_tokens', 0):>11} "
        f"{usage.get('output_tokens', 0):>8}  {text[:40]!r}"
    )

model                        latency  reason tok  out tok  answer
------------------------------------------------------------------------------------------------------------


openai.gpt-oss-20b             1.39s          24       79  'Batching improves GPU inference throughp'


openai.gpt-oss-120b            2.10s           6       55  'Batching lets the GPU process many input'


openai.gpt-5.4                 1.23s           0       34  'Batching improves GPU inference throughp'


openai.gpt-5.5                 1.59s           0       41  'Batching improves GPU inference throughp'


openai.gpt-5.6-sol             1.46s           0       27  'Batching improves GPU inference throughp'


## 11b. Service tiers are not uniform in this family

`flex` and `priority` trade cost against queue priority — but the **gpt-5.x
models accept only `default`**, while gpt-oss accepts all three. Sending `flex`
to gpt-5.6 is a 400, so tier selection has to be model-aware.

In [22]:
print(f"{'model':30} " + "  ".join(f"{t:>9}" for t in ("default", "flex", "priority")))
print("-" * 64)
tier_support = {}
for model in (SOL, GPT55, GPT54, OSS120, OSS20):
    prefix = prefix_for(model)
    row, allowed = [], []
    for tier in ("default", "flex", "priority"):
        code, _ = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": "Reply OK",
                "max_output_tokens": 16,
                "service_tier": tier,
            },
            region=REGION,
        )
        row.append("ok" if code == 200 else str(code))
        if code == 200:
            allowed.append(tier)
    tier_support[model] = allowed
    print(f"{model:30} " + "  ".join(f"{v:>9}" for v in row))

print("\nallowed tiers per model:")
for model, allowed in tier_support.items():
    print(f"   {model:30} {allowed}")

model                            default       flex   priority
----------------------------------------------------------------


openai.gpt-5.6-sol                    ok        400        400


openai.gpt-5.5                        ok        400        400


openai.gpt-5.4                        ok        400        400


openai.gpt-oss-120b                   ok         ok         ok


openai.gpt-oss-20b                    ok         ok         ok

allowed tiers per model:
   openai.gpt-5.6-sol             ['default']
   openai.gpt-5.5                 ['default']
   openai.gpt-5.4                 ['default']
   openai.gpt-oss-120b            ['default', 'flex', 'priority']
   openai.gpt-oss-20b             ['default', 'flex', 'priority']


## 12. Production hardening

In [23]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "openai-gpt-samples",
        "tags": {"Application": "OpenAIGPTDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)


class GPTClient:
    """Production shape: correct path per model, retries, attribution, no retention."""

    # gpt-5.x accepts only the default tier; gpt-oss accepts flex/priority too.
    TIERED_MODELS = ("openai.gpt-oss",)

    def __init__(self, model=SOL, region=REGION, tier="default", project=None):
        self.model, self.region, self.project = model, region, project
        self.prefix = prefix_for(model)
        if tier != "default" and not model.startswith(self.TIERED_MODELS):
            print(
                f"[note] {model} only supports service_tier='default' "
                f"— ignoring requested '{tier}'"
            )
            tier = "default"
        self.tier = tier

    def ask(self, prompt, *, effort="low", max_output_tokens=512, verbosity=None):
        body = {
            "model": self.model,
            "input": prompt,
            "max_output_tokens": max(16, max_output_tokens),  # API minimum is 16
            "reasoning": {"effort": effort},
            "service_tier": self.tier,
            "store": False,  # opt out of the 30-day retention default
            # top_p deliberately omitted: rejected on gpt-5.6
        }
        if verbosity:
            body["text"] = {"verbosity": verbosity}
        headers = {"OpenAI-Project": self.project} if self.project else None
        code, data = post(
            f"{self.prefix}/responses", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data


# Asking for "flex" on gpt-5.6 is silently downgraded by the guard above.
bot = GPTClient(model=SOL, tier="flex", project=project_id)
out = bot.ask("Name one benefit of speculative decoding.", verbosity="low")
print("answer:", response_text(out)[:160])
print("tier  :", out.get("service_tier"))

# gpt-oss really does honour flex.
oss_bot = GPTClient(model=OSS120, tier="flex", project=project_id)
oss_out = oss_bot.ask("Name one benefit of batching.", max_output_tokens=120)
print("\ngpt-oss tier:", oss_out.get("service_tier"))

project: 200 proj_ioal46lq...
[note] openai.gpt-5.6-sol only supports service_tier='default' — ignoring requested 'flex'


answer: It reduces inference latency by generating and verifying multiple tokens in parallel.
tier  : default



gpt-oss tier: flex


In [24]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

archived: 200 archived


## Gotchas — OpenAI GPT on bedrock-mantle

| Gotcha | Detail |
|---|---|
| **Two path prefixes** | `gpt-5.*` → `/openai/v1`; `gpt-oss*` → bare `/v1` |
| gpt-5.6 output cap | `max_output_tokens` on Responses, `max_completion_tokens` on Chat Completions. **`max_tokens` is rejected outright** — a 400 that looks like a missing API (§2b) |
| `top_p` | Acceptance differs per model — probe it (§3) rather than assuming |
| `max_output_tokens` | Minimum **16** |
| `reasoning.effort` | `none`/`low`/`medium`/`high`; **`minimal` rejected** |
| Reasoning replay | Send final answers only in multi-turn history |
| `store=False` | Blocks `previous_response_id` chaining (404) |
| Region | Availability differs per model and per endpoint, and moves. §10 probes it; the model card's regional table is authoritative |
| `service_tier` | gpt-5.x accepts **`default` only**; `flex`/`priority` → 400. gpt-oss accepts all |
| Web Search | Only this family — see `02-web-search-and-grounding.ipynb` |
| `max_tool_calls` | Accepted and **not enforced** — cap tool calls in your own loop (`03` §4) |
| `reasoning.effort` on easy tasks | Spends 0 reasoning tokens at every level; effort is a ceiling, not a floor |

## Next
- `02-web-search-and-grounding.ipynb` — the feature unique to this family
- `03-tools-and-structured-output.ipynb` · `04-prompt-caching-and-cost.ipynb`
- `05-server-side-tools-and-fine-tuning.ipynb`

## Also on `bedrock-runtime`? GPT-5.6 — yes, on all three APIs

This is the clearest example in the collection of why these notebooks probe instead
of assert. Until August 2026 the GPT-5.x models were `bedrock-mantle` only, and this
section said so. Then [the 17 August 2026
launch](https://aws.amazon.com/about-aws/whats-new/2026/08/amazon-bedrock-cross-region-openai-v2/)
put the GPT-5.6 family (`sol`, `terra`, `luna`) on `bedrock-runtime` — with the
**Responses, Chat Completions *and* Converse** APIs — plus cross-Region inference.
The sentence became false without anything in the notebook changing.

Two things follow, and they are worth separating:

- **Cross-Region inference is now mandatory here.** On `bedrock-runtime` this model
  has no in-Region option at all, so every call names an inference profile.
- **The OpenAI-shaped APIs are not exclusive to `bedrock-mantle` any more.** The same
  Responses payload works on `bedrock-runtime` at
  `https://bedrock-runtime.{region}.amazonaws.com/openai/v1`, with either SigV4 or
  a Bedrock API key — so the OpenAI SDK works there unchanged.

The model card carries a tip worth repeating: *"Whenever possible, we recommend using
the `bedrock-runtime` endpoint for new applications."* Earlier GPT-5.x models may
still be mantle-only, so the probe below asks the live catalogues rather than
trusting this paragraph.


In [25]:
from bedrock import endpoints_for, inference_profiles, runtime_models

MODEL = "openai.gpt-5.6-sol"
print(f"{MODEL} -> {endpoints_for(MODEL)}")

print("\nopenai.* in the bedrock-runtime catalogue today:")
for model in sorted(m for m in runtime_models() if m.startswith("openai.")):
    print("   ", model)

print(f"\nInference profiles that carry {MODEL}:")
for profile in sorted(p for p in inference_profiles() if p.endswith(MODEL)):
    print("   ", profile)


openai.gpt-5.6-sol -> {'mantle': True, 'runtime': True}

openai.* in the bedrock-runtime catalogue today:
    openai.gpt-5.6-luna
    openai.gpt-5.6-sol
    openai.gpt-5.6-terra
    openai.gpt-oss-120b-1
    openai.gpt-oss-20b-1
    openai.gpt-oss-safeguard-120b
    openai.gpt-oss-safeguard-20b

Inference profiles that carry openai.gpt-5.6-sol:
    global.openai.gpt-5.6-sol
    us.openai.gpt-5.6-sol


### The bare model ID is rejected, and the error misdirects

Converse will not take `openai.gpt-5.6-sol` as-is. It answers:

> `ValidationException` — Invocation of model ID ... with on-demand throughput
> isn't supported. Retry your request with the ID or ARN of an inference profile
> that contains this model.

That reads like a permissions or entitlement problem. It is not, and it is not a
quirk either — the model card lists the in-Region endpoint URL for the
`bedrock-runtime` row as **"Not supported"**. In-Region invocation simply is not
offered for this model on this endpoint, so the ID must carry a Region scope:
`us.` for the geographic profile, `global.` for the global one.

Most Claude models behave the same way, which is why
[`resolve_runtime_id()`](../_shared/bedrock.py) exists: it looks the model up in the
profile list and returns the form Converse will accept.

The cell below passes `resolve=False` to defeat that helper, so you can see the raw
failure and the fix side by side.


In [26]:
from bedrock import converse, resolve_runtime_id

QUESTION = [{"role": "user", "content": [{"text": "In one sentence: what is idempotency?"}]}]

print("resolve_runtime_id() picks:", resolve_runtime_id(MODEL), "\n")

for model_id in (MODEL, f"us.{MODEL}", f"global.{MODEL}"):
    # resolve=False so the bare ID really is sent bare.
    text, response = converse(model_id, QUESTION, max_tokens=80, resolve=False)
    if response.get("error"):
        print(f"{model_id:28} {response['error']['code']}")
        print(f"{'':28} {response['error']['message'][:96]}...")
    else:
        print(f"{model_id:28} ok, {response['usage']['outputTokens']} output tokens")
        print(f"{'':28} {text}")


resolve_runtime_id() picks: us.openai.gpt-5.6-sol 



openai.gpt-5.6-sol           ValidationException
                             Invocation of model ID openai.gpt-5.6-sol with on-demand throughput isn’t supported. Retry your ...


us.openai.gpt-5.6-sol        ok, 25 output tokens
                             Idempotency is the property of an operation that produces the same result whether performed once or multiple times.


global.openai.gpt-5.6-sol    ok, 25 output tokens
                             Idempotency is the property of an operation that produces the same result whether performed once or multiple times.


### Choosing between `us.` and `global.`

Both work. They are not interchangeable, and the difference is not performance:

| | `us.` — geographic | `global.` — global |
|---|---|---|
| Where inference runs | a commercial Region **inside the US geography** | **any** supported commercial Region worldwide |
| Reach for it when | you have data-residency obligations | you do not, and you want the most capacity |
| Throughput | good | highest — the widest pool during demand spikes |
| Token price | standard | **~10% lower**, per the cross-Region inference guide |

Neither changes where your data is *stored*. Cross-Region inference moves the
transient computation only: invocation logs, knowledge bases and configuration stay
in the source Region, traffic never leaves the AWS network, and it is encrypted in
transit. What moves is the inference itself — which is exactly what a data-residency
reviewer will ask about, so `global.` is the one that needs a sign-off, not `us.`.

Three operational details that catch people out, all from the [cross-Region inference
guide](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html):

- **Service control policies.** Geographic profiles need every destination Region in
  the profile allowed. Global profiles need `"aws:RequestedRegion": "unspecified"`
  allowed — an SCP that enumerates Regions will block `global.` even though the model
  is enabled. This is the most common reason `global.` fails in an Organization.
- **Destination Regions do not need to be enabled in your account.** Cross-Region
  inference can route to Regions you have never turned on, and that is by design.
- **You can prove where a request ran.** CloudTrail logs it in the source Region;
  read `additionalEventData.inferenceRegion`. Worth knowing before you promise a
  reviewer anything about residency.

There is no extra routing charge, and the rate is set by the Region you call from,
not the one that serves the request. Prices move — the [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-56-sol.html)
carries the current In-Region, Geo and Global tables side by side.


### Three APIs, one model — what actually differs

`bedrock-runtime` serves this model on Responses, Chat Completions and Converse. The
model is identical; what changes is the envelope you write and the reply you parse.

| | Responses | Chat Completions | Converse |
|---|---|---|---|
| Path / call | `POST /openai/v1/responses` | `POST /openai/v1/chat/completions` | `bedrock-runtime` SDK `converse()` |
| Input field | `input` (string or list) | `messages` | `messages` with typed content blocks |
| Output cap | `max_output_tokens` | `max_completion_tokens` | `inferenceConfig.maxTokens` |
| Where the answer is | `output[]` → `content[]` → `output_text` | `choices[0].message.content` | `output.message.content[]` → `text` |
| Reasoning **and** tools together | yes | **no** — see below | tools yes, trace not returned |
| Auth | SigV4 | SigV4 | SigV4 via the SDK |

Reach for Responses if you are writing new OpenAI-shaped code, Chat Completions if you
have an existing corpus of it, and Converse if you want one request shape across every
provider on Bedrock.

The cell below sends the same question four ways — once to `bedrock-mantle` and three
times to `bedrock-runtime` — so the shape differences are visible next to each other.


In [27]:
import boto3
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

RUNTIME_BASE = f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1"
PROFILE = f"us.{MODEL}"


def runtime_openai_post(path: str, body: dict, stream: bool = False):
    """POST an OpenAI-shaped payload to bedrock-runtime, signed with SigV4.

    Note the service name stays "bedrock" even though the host is bedrock-runtime.
    SigV4 is one of two options here; the next section shows the other.
    """
    payload = json.dumps(body)
    signed = AWSRequest(
        method="POST",
        url=f"{RUNTIME_BASE}{path}",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    SigV4Auth(
        boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials(),
        "bedrock",
        REGION,
    ).add_auth(signed)
    reply = requests.post(
        f"{RUNTIME_BASE}{path}",
        headers=dict(signed.headers),
        data=payload,
        timeout=120,
        stream=stream,
    )
    return reply


def body_of(reply) -> dict:
    """Parsed JSON, or {} - an error reply is not always JSON, and a cell that
    crashes on that teaches nothing. Mirrors how `post` behaves."""
    try:
        return reply.json()
    except ValueError:
        return {}


QUESTION = "In one sentence: what is idempotency?"

# 1. bedrock-mantle, Responses - the bearer-token path used everywhere above.
code, mantle = post(f"{GPT5_PREFIX}/responses", {"model": MODEL, "input": QUESTION, "max_output_tokens": 60})
print(f"mantle   /openai/v1/responses         {code}  {response_text(mantle)[:78]}")

# 2. bedrock-runtime, Responses - same body, SigV4 instead of a token.
reply = runtime_openai_post("/responses", {"model": PROFILE, "input": QUESTION, "max_output_tokens": 60})
print(f"runtime  /openai/v1/responses         {reply.status_code}  {response_text(reply.json())[:78]}")

# 3. bedrock-runtime, Chat Completions - messages[] and max_completion_tokens.
reply = runtime_openai_post(
    "/chat/completions",
    {"model": PROFILE, "messages": [{"role": "user", "content": QUESTION}], "max_completion_tokens": 60},
)
print(f"runtime  /openai/v1/chat/completions  {reply.status_code}  "
      f"{reply.json()['choices'][0]['message']['content'][:78]}")

# 4. bedrock-runtime, Converse - typed content blocks, via the AWS SDK.
text, _ = converse(MODEL, [{"role": "user", "content": [{"text": QUESTION}]}], max_tokens=60)
print(f"runtime  Converse                     200  {text[:78]}")


mantle   /openai/v1/responses         200  Idempotency is the property whereby performing an operation multiple times has


runtime  /openai/v1/responses         200  Idempotency is the property of an operation that produces the same result whet


runtime  /openai/v1/chat/completions  200  Idempotency is the property of an operation whereby repeating it produces the 


runtime  Converse                     200  Idempotency is the property where performing an operation multiple times has t


### Two ways to authenticate, and the easy migration path

`bedrock-runtime`'s `/openai/v1` accepts **either** credential style, and which you
pick changes how much code you write:

| | Bedrock API key as bearer token | SigV4 |
|---|---|---|
| Works with the OpenAI SDK | **yes**, unchanged | no — the SDK cannot sign |
| Code change from `bedrock-mantle` | **the base URL and the model ID** | swap auth for signing |
| Credential lifecycle | short-term key, needs refreshing | ordinary AWS credential chain, nothing to mint |
| Reach for it when | migrating existing OpenAI-SDK code | you already have AWS credentials in process |

So the shortest migration is genuinely two edits: point `base_url` at
`https://bedrock-runtime.{region}.amazonaws.com/openai/v1` and name the inference
profile instead of the bare model. SigV4 is the better long-term shape — no token to
mint, rotate or leak — but it is not required, and the cell below shows both.

Stored responses have a lifecycle here too, on the same base URL:
`GET /openai/v1/responses/{id}` retrieves one, `POST .../{id}/cancel` cancels an
in-flight one, and `DELETE .../{id}` removes it.

One IAM subtlety worth knowing before you write a policy. Creating a response
authorises **two** resources: `bedrock:InvokeModel` on the inference target, as any
inference call does, **and** `bedrock:InvokeModel` on your account's default project.
A policy scoped to the model ARN alone will fail. Retrieve, cancel and delete
authorise `bedrock:GetInvoke`, `bedrock:CancelInvoke` and `bedrock:DeleteInvoke` on
the project. Response IDs are not IAM resources.


In [28]:
from openai import OpenAI

from bedrock import token

# 1. The OpenAI SDK, pointed at bedrock-runtime with a Bedrock API key. No signing.
runtime_sdk = OpenAI(api_key=token(REGION), base_url=RUNTIME_BASE)
ASK = "In one word: what does idempotent mean?"

# Budget generously. This model reasons first, and a tight cap is spent entirely on
# reasoning: status comes back "incomplete", output holds a reasoning item and no
# message, and the text is empty. Printing `status` keeps that visible instead of
# looking like a broken cell.
sdk_reply = runtime_sdk.responses.create(model=PROFILE, input=ASK, max_output_tokens=200)
print(f"OpenAI SDK + bearer token  [{sdk_reply.status}] {response_text(sdk_reply.model_dump())[:60]}")

# 2. The same call signed with SigV4, for code that already holds AWS credentials.
signed = body_of(
    runtime_openai_post("/responses", {"model": PROFILE, "input": ASK, "max_output_tokens": 200})
)
print(f"SigV4                      [{signed.get('status')}] {response_text(signed)[:60]}")

# 3. Stored-response lifecycle: create, retrieve, delete.
created = body_of(runtime_openai_post(
    "/responses", {"model": PROFILE, "input": "Remember the number 9.", "max_output_tokens": 24, "store": True}
))
fetched = runtime_sdk.responses.retrieve(created["id"])
runtime_sdk.responses.delete(created["id"])
# Prove the delete rather than trusting its return value: retrieval must now fail.
try:
    runtime_sdk.responses.retrieve(created["id"])
    gone = "still retrievable"
except Exception as exc:
    gone = type(exc).__name__
print(f"\nstore -> retrieve -> delete: retrieved status={fetched.status}, after delete={gone}")


OpenAI SDK + bearer token  [completed] Repeatable


SigV4                      [completed] Repeatable.



store -> retrieve -> delete: retrieved status=completed, after delete=NotFoundError


### Tool use: the one place the APIs genuinely disagree

Every path supports function tools, but **Chat Completions refuses tools and reasoning
in the same request**. Ask for both and you get a 400 that, unusually, tells you
exactly what to do:

> Function tools with `reasoning_effort` are not supported for
> `us.openai.gpt-5.6-sol` in `/v1/chat/completions`. To use function tools, use
> `/v1/responses` or set `reasoning_effort` to `'none'`.

So on Chat Completions you choose: reasoning, or tools. Responses and Converse give
you both. If you are moving an agent onto this model, that single constraint is the
strongest argument for Responses over Chat Completions.

Note also that the three APIs return a tool call in three different shapes, so the
parsing is never portable even when the request nearly is.


In [29]:
STOCK_TOOL_RESPONSES = [
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Current share price for a ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {"ticker": {"type": "string"}},
            "required": ["ticker"],
        },
    }
]
# Chat Completions nests the same thing one level deeper, under "function".
STOCK_TOOL_CHAT = [{"type": "function", "function": STOCK_TOOL_RESPONSES[0].copy()}]
del STOCK_TOOL_CHAT[0]["function"]["type"]

ASK = "What is AMZN trading at? Use the tool."

# Responses: tools and reasoning coexist.
reply = runtime_openai_post(
    "/responses", {"model": PROFILE, "input": ASK, "tools": STOCK_TOOL_RESPONSES, "max_output_tokens": 300}
)
items = [item.get("type") for item in body_of(reply).get("output", [])]
calls = [(i.get("name"), i.get("arguments")) for i in body_of(reply).get("output", []) if i.get("type") == "function_call"]
print(f"responses         {reply.status_code}  output items={items}\n                       calls={calls}")

# Chat Completions, reasoning left at its default: rejected.
reply = runtime_openai_post(
    "/chat/completions",
    {"model": PROFILE, "messages": [{"role": "user", "content": ASK}], "tools": STOCK_TOOL_CHAT,
     "max_completion_tokens": 300},
)
print(f"\nchat/completions  {reply.status_code}  {body_of(reply)['error']['message']}")

# Chat Completions with reasoning switched off: accepted.
reply = runtime_openai_post(
    "/chat/completions",
    {"model": PROFILE, "messages": [{"role": "user", "content": ASK}], "tools": STOCK_TOOL_CHAT,
     "reasoning_effort": "none", "max_completion_tokens": 300},
)
choice = body_of(reply)["choices"][0]
print(f'\nchat/completions  {reply.status_code}  reasoning_effort="none" -> finish={choice["finish_reason"]}')
for call in choice["message"].get("tool_calls") or []:
    print(f"                       {call['function']['name']}({call['function']['arguments']})")


responses         200  output items=['function_call']
                       calls=[('get_stock_price', '{"ticker":"AMZN"}')]



chat/completions  400  Function tools with reasoning_effort are not supported for us.openai.gpt-5.6-sol in /v1/chat/completions. To use function tools, use /v1/responses or set reasoning_effort to 'none'.



chat/completions  200  reasoning_effort="none" -> finish=tool_calls
                       get_stock_price({"ticker":"AMZN"})


### What else carries over to `bedrock-runtime`

Four things people assume are `bedrock-mantle` features, probed here rather than
assumed:

- **Streaming.** `"stream": true` on Responses returns the same typed
  `response.*` SSE events.
- **Prompt caching.** Supported on the Responses API. `usage.input_tokens_details`
  reports `cache_write_tokens` on the first call and `cached_tokens` on a repeat.
- **Server-side state.** `store: true` plus `previous_response_id` chains turns
  without you resending history.
- **Structured outputs.** Both OpenAI APIs accept a JSON schema and honour it —
  `text.format` on Responses, `response_format` on Chat Completions.

The last one is worth flagging: the model card lists *Structured outputs* under **not
supported** for `bedrock-runtime`, and the API plainly accepts them. Either the card
means the Bedrock-native feature rather than the OpenAI schema field, or it is behind.
The cell below is the evidence; trust it over this paragraph and over the card, and
re-run it before you rely on the behaviour.


In [30]:
# 1. Streaming.
reply = runtime_openai_post(
    "/responses", {"model": PROFILE, "input": "Count to three.", "max_output_tokens": 40, "stream": True},
    stream=True,
)
events = []
for line in reply.iter_lines():
    if line and line.startswith(b"data: "):
        try:
            events.append(json.loads(line[6:]).get("type"))
        except json.JSONDecodeError:
            pass
print(f"streaming         {reply.status_code}  {len(events)} events, e.g. {sorted(set(events))[:3]}")

# 2. Prompt caching. The system block has to be big enough to be worth caching.
#    Note "first call" is only cold the first time this notebook ever runs against
#    this prefix: the entry survives between runs, so on a re-run both calls read
#    from cache. Report what happened rather than asserting a write.
PRIMER = [{"role": "system", "content": "You are a terse assistant. " * 400},
          {"role": "user", "content": "Say ok."}]
cache_rows = []
for attempt in (1, 2):
    reply = runtime_openai_post("/responses", {"model": PROFILE, "input": PRIMER, "max_output_tokens": 20})
    detail = body_of(reply)["usage"]["input_tokens_details"]
    cache_rows.append((detail.get("cache_write_tokens") or 0, detail.get("cached_tokens") or 0))
    print(f"caching call {attempt}    {reply.status_code}  "
          f"cache_write={detail.get('cache_write_tokens')} cached={detail.get('cached_tokens')}")
if cache_rows[0][0] and cache_rows[1][1]:
    print("                       -> a cold write then a warm read, as designed")
elif cache_rows[1][1]:
    print("                       -> both calls read from cache: the prefix was")
    print("                          already warm from an earlier run. Caching works")
    print("                          across processes, which is the point of it.")
else:
    print("                       -> no cache activity; the prefix may be under the")
    print("                          1,024-token minimum")

# 3. Server-side state.
first = body_of(runtime_openai_post(
    "/responses", {"model": PROFILE, "input": "Remember the number 7.", "max_output_tokens": 30, "store": True}
))
follow = runtime_openai_post(
    "/responses",
    {"model": PROFILE, "input": "Which number did I ask you to remember?",
     "previous_response_id": first["id"], "max_output_tokens": 30},
)
print(f"\nserver-side state 200  chained reply: {response_text(body_of(follow))[:60]}")

# 4. Structured outputs, both OpenAI APIs.
PLACE_SCHEMA = {
    "type": "object",
    "properties": {"city": {"type": "string"}, "country": {"type": "string"}},
    "required": ["city", "country"],
    "additionalProperties": False,
}
reply = runtime_openai_post(
    "/responses",
    {"model": PROFILE, "input": "Rome, Italy", "max_output_tokens": 100,
     "text": {"format": {"type": "json_schema", "name": "place", "schema": PLACE_SCHEMA, "strict": True}}},
)
print(f"\nstructured (resp) {reply.status_code}  {response_text(body_of(reply))[:60]}")

reply = runtime_openai_post(
    "/chat/completions",
    {"model": PROFILE, "messages": [{"role": "user", "content": "Rome, Italy"}],
     "max_completion_tokens": 100, "reasoning_effort": "none",
     "response_format": {"type": "json_schema",
                         "json_schema": {"name": "place", "schema": PLACE_SCHEMA, "strict": True}}},
)
print(f"structured (chat) {reply.status_code}  {body_of(reply)['choices'][0]['message']['content'][:60]}")


streaming         200  14 events, e.g. ['response.completed', 'response.content_part.added', 'response.content_part.done']


caching call 1    200  cache_write=2412 cached=0


caching call 2    200  cache_write=0 cached=2412
                       -> a cold write then a warm read, as designed



server-side state 200  chained reply: 7



structured (resp) 200  {"city":"Rome","country":"Italy"}


structured (chat) 200  {"city":"Rome","country":"Italy"}


### Where `bedrock-runtime` behaves differently, and one place the docs are wrong

Same request format, but not the same behaviour. These are the differences that break
working code, rather than the ones you would notice in a feature table:

- **`background=true` is rejected.** Runtime answers 400 *"The background parameter
  is not supported"*; `bedrock-mantle` accepts it. Asynchronous work stays on mantle.
  `store` is unaffected and still defaults to `true`, so stored multi-turn
  conversations are fine.
- **Server-side tools are absent**, web search included. Client-side function tools
  work on both.
- **Application inference profiles are rejected.** System, geographic and global
  profiles work; an application profile as the inference target returns 400.
- **Guardrails do not apply to the Responses API here.** To guardrail a GPT model on
  `bedrock-runtime`, call Converse instead.
- **A stored response belongs to the Region that served it.** Retrieve, cancel,
  delete and `previous_response_id` are all handled by that Region, and an ID that
  cannot be found returns the same 404 whether it never existed, belongs to another
  account, or was never stored.

**And one correction to the documentation.** The user guide says `model` may be
omitted on a follow-up that carries `previous_response_id` on `bedrock-mantle`, and
that requiring it is a `bedrock-runtime` difference. Both endpoints require it — the
cell below sends the same follow-up to each and both return 400. Treat `model` as
mandatory everywhere and the difference disappears.


In [31]:
# background=true: accepted on mantle, rejected on runtime.
runtime_bg = runtime_openai_post(
    "/responses", {"model": PROFILE, "input": "Write a haiku.", "max_output_tokens": 40, "background": True}
)
mantle_bg, mantle_body = post(
    f"{GPT5_PREFIX}/responses",
    {"model": MODEL, "input": "Write a haiku.", "max_output_tokens": 40, "background": True},
)
print(f'background=true   runtime {runtime_bg.status_code}: '
      f'{body_of(runtime_bg).get("error", {}).get("message", "") or runtime_bg.text[:56] or "(empty body)"}')
print(f'                  mantle  {mantle_bg}: {"accepted" if mantle_bg == 200 else err(mantle_body)[:56]}')

# `model` on a follow-up: the docs say mantle lets you omit it. Send it to both.
seed_runtime = runtime_openai_post(
    "/responses", {"model": PROFILE, "input": "Remember the number 9.", "max_output_tokens": 24, "store": True}
)
seed_runtime = body_of(seed_runtime)
_, seed_mantle = post(
    f"{GPT5_PREFIX}/responses",
    {"model": MODEL, "input": "Remember the number 9.", "max_output_tokens": 24, "store": True},
)

no_model_runtime = runtime_openai_post(
    "/responses", {"input": "Which number?", "previous_response_id": seed_runtime["id"], "max_output_tokens": 24}
)
no_model_mantle, no_model_body = post(
    f"{GPT5_PREFIX}/responses",
    {"input": "Which number?", "previous_response_id": seed_mantle["id"], "max_output_tokens": 24},
)
print(f'\nfollow-up, no model   runtime {no_model_runtime.status_code} | mantle {no_model_mantle}')
print(f'                      mantle says: {err(no_model_body)[:70]}')
print("Both reject it, so the documented difference does not exist. Always send `model`.")

# An ID that was never stored: 404, and deliberately indistinguishable from an ID
# that belongs to someone else. Retrieve raises, so catch it rather than crash.
try:
    runtime_sdk.responses.retrieve("resp_thisidwillneverexist000")
    print("\nunknown response id   unexpectedly succeeded")
except Exception as exc:  # openai.NotFoundError
    print(f"\nunknown response id   {type(exc).__name__}: {str(exc)[:66]}")


background=true   runtime 400: The background parameter is not supported.
                  mantle  200: accepted



follow-up, no model   runtime 400 | mantle 400
                      mantle says: Missing required parameter: 'model'.
Both reject it, so the documented difference does not exist. Always send `model`.



unknown response id   NotFoundError: Error code: 404 - {'error': {'message': 'The requested resource co


### Converse: one shape for every provider

Converse is the reason to be on `bedrock-runtime` if you run more than one model
family. The request and reply shape is identical across providers, so an agent loop
written for Claude or Nova runs against this model unchanged — no new parsing.

Tool use returns an ordinary `toolUse` block. The reasoning trace does **not** come
back: `content` carries a `text` block and nothing else, so the thinking is billed and
unreadable, the same trade the `/v1` families make on Chat Completions.


In [32]:
from bedrock import converse_reasoning, converse_tool_uses

STOCK_TOOL = [
    {
        "toolSpec": {
            "name": "get_stock_price",
            "description": "Current share price for a ticker symbol.",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {"ticker": {"type": "string"}},
                    "required": ["ticker"],
                }
            },
        }
    }
]

# resolve defaults to True from here on, so the bare ID is fine to pass.
_, tooled = converse(
    MODEL,
    [{"role": "user", "content": [{"text": "What is AMZN trading at? Use the tool."}]}],
    max_tokens=300,
    tools=STOCK_TOOL,
)
print("stopReason:", tooled.get("stopReason"))
for call in converse_tool_uses(tooled):
    print("   tool:", call["name"], "input:", call["input"])

answer, reasoned = converse(
    MODEL,
    [{"role": "user", "content": [{"text": "A bat and ball cost $1.10. The bat costs $1.00 more than the ball. What does the ball cost?"}]}],
    max_tokens=400,
)
blocks = [key for block in reasoned["output"]["message"]["content"] for key in block]
print(f"\ncontent blocks: {blocks}")
print(f"reasoning returned: {len(converse_reasoning(reasoned))} chars")
print("answer:", answer.strip()[:160])


stopReason: tool_use
   tool: get_stock_price input: {'ticker': 'AMZN'}



content blocks: ['text']
reasoning returned: 0 chars
answer: The ball costs **$0.05** (5 cents).

The bat costs $1.05, so together they cost $1.10.


### Choosing an endpoint for this model

| | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| APIs | Responses · Chat Completions · Converse | Responses · Chat Completions |
| Auth | SigV4 **or** Bedrock API key | Bedrock API key |
| OpenAI SDK works | **yes**, with the API key | yes |
| Cross-Region inference | **required** (geo or global) | not supported |
| Regional reach | wide | narrow |
| Streaming · prompt caching · server-side state · structured outputs | yes (caching is Responses-only) | yes |
| `background=true` | **no** — 400 | yes |
| Server-side tools (web search, code interpreter) | **no** | yes |
| Application inference profiles | **no** — 400 | n/a |
| Guardrails | Converse only | — |
| Invocation logs · CloudWatch metrics · Cost Explorer per-model lines | **yes** | limited |
| One request shape across providers | **yes**, via Converse | no |

The trade in one line: `bedrock-mantle` for server-side tools and background jobs,
`bedrock-runtime` for AWS-native governance, cross-Region throughput and Converse
portability. The public model card recommends `bedrock-runtime` for new applications.

Feature matrices move faster than notebooks, and this section found the card and the
docs wrong twice — structured outputs above, and the `model`-on-follow-up rule. Probe
before you commit.
